In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time

In [28]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [5]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


In [6]:
KEYWORDS = [
    "education",
    'india'
]

In [7]:
import requests
from bs4 import BeautifulSoup

def get_article_links():
    url = "https://news.google.com/rss/search?q=education+india"
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "xml")

    links = []
    for item in soup.find_all("item"):
        links.append(item.link.text)

    return links

In [8]:
l=get_article_links()

In [9]:
print(len(l))

100


In [10]:
from newspaper import Article

def extract_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except Exception as e:
        print(f"Failed: {url}")
        return None

In [11]:
def is_relevant(text):
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in KEYWORDS)

In [12]:
def get_relevant_articles():
    links = get_article_links()
    results = []

    for url in links:
        driver.get(url)
        time.sleep(1)  # wait for redirects + page load

        # optional: get final resolved URL
        final_url = driver.current_url
        print(f"Processing: {final_url}")
        text = extract_article(final_url)

        if text and is_relevant(text):
            results.append({
                "url": final_url,
                "text": text  # preview
            })

    return results

In [13]:
articles = get_relevant_articles()



Processing: https://www.pib.gov.in/PressReleasePage.aspx?PRID=2250371
Failed: https://www.pib.gov.in/PressReleasePage.aspx?PRID=2250371
Processing: https://www.thehindu.com/education/reimagining-automation-education-for-indias-industrial-growth/article70837495.ece
Processing: https://www.ndtv.com/offbeat/indian-student-shares-france-education-costs-internet-reacts-to-rs-45-lakh-breakdown-11336048
Processing: https://education.economictimes.indiatimes.com/news/international/indian-students-in-ireland-rose-from-700-to-over-9000-over-past-decade-report/130157192
Processing: https://economictimes.indiatimes.com/industry/services/education/why-global-students-are-choosing-india-to-build-d2c-startups/articleshow/130141618.cms?from=mdr
Failed: https://economictimes.indiatimes.com/industry/services/education/why-global-students-are-choosing-india-to-build-d2c-startups/articleshow/130141618.cms?from=mdr
Processing: https://www.indiatoday.in/amp/education-today/story/cbse-three-language-policy-c

In [20]:
type(articles)
articles

[{'url': 'https://www.thehindu.com/education/reimagining-automation-education-for-indias-industrial-growth/article70837495.ece',
  'text': 'India’s goal of becoming the third largest global economy depends on how well we grow our manufacturing capacity. The efficiency and productivity of the nation’s factories will be key factor in achieving this vision. However, the critical enabler of this shift will be the automation technology, which integrates systems to run machines effortlessly, manages water distribution of right purity on time, every time, regulates power flow in the electrical grid and controls streetlights in the cities.\n\n(Sign up for THEdge, The Hindu’s weekly education newsletter.)'},
 {'url': 'https://www.ndtv.com/offbeat/indian-student-shares-france-education-costs-internet-reacts-to-rs-45-lakh-breakdown-11336048',
  'text': 'An Indian student, named Raine, recently shared their experience of studying in France, revealing the costs involved. The online users reacted to

In [15]:
for i, art in enumerate(articles):

    file_content = art['text']
    file_path = fr"C:\Users\amolr\OneDrive\Desktop\education\output{i}.txt"
    with open(file_path, "w") as text_file:
        text_file.write(file_content)


In [21]:
import pandas as pd
import numpy as np

In [22]:
articles

[{'url': 'https://www.thehindu.com/education/reimagining-automation-education-for-indias-industrial-growth/article70837495.ece',
  'text': 'India’s goal of becoming the third largest global economy depends on how well we grow our manufacturing capacity. The efficiency and productivity of the nation’s factories will be key factor in achieving this vision. However, the critical enabler of this shift will be the automation technology, which integrates systems to run machines effortlessly, manages water distribution of right purity on time, every time, regulates power flow in the electrical grid and controls streetlights in the cities.\n\n(Sign up for THEdge, The Hindu’s weekly education newsletter.)'},
 {'url': 'https://www.ndtv.com/offbeat/indian-student-shares-france-education-costs-internet-reacts-to-rs-45-lakh-breakdown-11336048',
  'text': 'An Indian student, named Raine, recently shared their experience of studying in France, revealing the costs involved. The online users reacted to

In [24]:
df = pd.DataFrame({'url':[a1['url'] for a1 in articles], 'preview':[a1['text'] for a1 in articles]})
df

,url,preview
0,https://www.thehindu.com/education/reimagining...,India’s goal of becoming the third largest glo...
1,https://www.ndtv.com/offbeat/indian-student-sh...,"An Indian student, named Raine, recently share..."
2,https://education.economictimes.indiatimes.com...,Advt\n\nAdvt\n\nJoin the community of 2M+ indu...
3,https://www.indiatoday.in/amp/education-today/...,It feels like Deja vu for the Indian education...
4,https://www.insightsonindia.com/2026/04/10/mak...,Source: TH\n\nSubject: Education\n\nContext: T...
...,...,...
62,https://education.economictimes.indiatimes.com...,By Neerja Birla.\n\nAdvt\n\nAdvt\n\nMoving fro...
63,https://blog.google/intl/en-in/powering-indias...,A recent survey with Ipsos revealed something ...
64,https://www.orfonline.org/research/the-nep-202...,"Editor’s Note\n\nBy 2020, when the new Nationa..."
65,https://www.worldbank.org/en/news/press-releas...,"Washington, Nov. 25, 2025—The World Bank Board..."


In [25]:
df.to_excel(r'C:\Users\amolr\OneDrive\Desktop\education\news_article.xlsx')

In [54]:
df = df.str.lower()

In [55]:
df=df.apply(lambda x: re.sub('[^a-zA-z0-9]+',' ',x))

In [35]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\amolr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [56]:
stopword = set(stopwords.words('english'))

In [93]:
custom_words = ["education", "student", "india", "school"]

In [94]:
stopword = stopword.union(custom_words)

In [95]:
## removing stopwards and whitespace
df = df.apply(lambda x: ' '.join([y for y in x.split() if y not in stopword]))
df=df.apply(lambda x: ' '.join(x.split()))

In [96]:
df

0     goal becoming third largest global economy dep...
1     indian named raine recently shared experience ...
2     advt advt join community 2m industry professio...
3     feel like deja vu indian system something deep...
4     source th subject context gross enrolment rati...
                            ...                        
62    neerja birla advt advt moving reactive prevent...
63    recent survey ipsos revealed something remarka...
64    editor note 2020 new national policy nep enact...
65    washington nov 25 2025 world bank board execut...
66    survive thrive chaos unleashed trump subscribe...
Name: preview, Length: 67, dtype: object

In [97]:
lemmatizer = WordNetLemmatizer()

In [98]:
def lemmatize_words(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

In [99]:

df = df.apply(lambda  x: lemmatize_words(x))

In [100]:
df

0     goal becoming third largest global economy dep...
1     indian named raine recently shared experience ...
2     advt advt join community 2m industry professio...
3     feel like deja vu indian system something deep...
4     source th subject context gross enrolment rati...
                            ...                        
62    neerja birla advt advt moving reactive prevent...
63    recent survey ipsos revealed something remarka...
64    editor note 2020 new national policy nep enact...
65    washington nov 25 2025 world bank board execut...
66    survive thrive chaos unleashed trump subscribe...
Name: preview, Length: 67, dtype: object

In [101]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_df=0.85,
    min_df=10,
    ngram_range=(1,2))
X = vectorizer.fit_transform(df).toarray()

In [102]:
X

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 5,  0,  0, ...,  0,  0,  0],
       [ 1,  0,  0, ...,  1,  0,  0],
       ...,
       [ 0,  0, 16, ...,  4,  0,  0],
       [ 2,  0,  0, ...,  4,  0,  0],
       [ 0,  0,  1, ...,  1,  0,  0]], shape=(67, 340))

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 5

lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(X)

,n_components,5
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [104]:
features = vectorizer.get_feature_names_out()

In [105]:
features

array(['000', '10', '2020', '2023', '2024', '2025', '2026', '25', '30',
       '50', 'academic', 'access', 'according', 'across', 'add', 'added',
       'address', 'advertisement', 'ai', 'aim', 'aimed', 'aligned',
       'along', 'alongside', 'already', 'also', 'among', 'analysis',
       'another', 'application', 'approach', 'april', 'area', 'around',
       'available', 'based', 'become', 'becoming', 'benefit', 'better',
       'beyond', 'board', 'bridge', 'broader', 'build', 'building',
       'built', 'capacity', 'career', 'case', 'central', 'challenge',
       'change', 'child', 'class', 'classroom', 'clear', 'collaboration',
       'college', 'come', 'commitment', 'community', 'comprehensive',
       'concern', 'continues', 'core', 'cost', 'could', 'country',
       'course', 'create', 'creating', 'critical', 'cultural',
       'curriculum', 'data', 'day', 'decision', 'degree', 'delhi',
       'designed', 'develop', 'development', 'digital', 'discussion',
       'district', 'dr',

In [106]:
def display_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx}:")
        print(" ".join([feature_names[i] 
                        for i in topic.argsort()[:-n_top_words - 1:-1]]))

feature_names = vectorizer.get_feature_names_out()
display_topics(lda, feature_names)


Topic 0:
nep learning institution 2020 nep 2020 policy system national world one

Topic 1:
language indian cost university year 2026 policy one global 10

Topic 2:
industry result academic digital learning year programme engineering infrastructure board

Topic 3:
outcome skill state government development national support private initiative also

Topic 4:
ai system indian knowledge partnership help innovation also governance university
